In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
from json import load as load_json

In [2]:
_Delay = 2.5

with open("config.json", "r") as f:
        headers = load_json(f)
headers

{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
 'Accept-Language': 'en-US,en;q=0.9',
 'Accept-Encoding': 'gzip, deflate, br',
 'Connection': 'keep-alive',
 'Upgrade-Insecure-Requests': '1',
 'Sec-Fetch-Dest': 'document',
 'Sec-Fetch-Mode': 'navigate',
 'Sec-Fetch-Site': 'none',
 'Sec-Fetch-User': '?1',
 'Cache-Control': 'max-age=0',
 'DNT': '1'}

In [4]:
with open("../../data/raw/NBA_per_game.txt", "r") as f:
    season_url = f.read().splitlines()

columns = [
        "ID",
        "Season",
        "Player Name",
        "Team",
        
        "Position",
        "Games Played (G)",
        "Games Started (GS)",
        "Minutes Played (MP)",
        "Field Goals Made (FG)",
        "Field Goal Attempts (FGA)",
        "Field Goal Percentage (FG%)",
        "3-Point Field Goals Made (3P)",
        "3-Point Field Goal Attempts (3PA)",
        "3-Point Field Goal Percentage (3P%)",
        "2-Point Field Goals Made (2P)",
        "2-Point Field Goal Attempts (2PA)",
        "2-Point Field Goal Percentage (2P%)",
        "Effective Field Goal Percentage (eFG%)",
        "Free Throws Made (FT)",
        "Free Throw Attempts (FTA)",
        "Free Throw Percentage (FT%)",
        "Offensive Rebounds (ORB)",
        "Offensive Rebound Percentage (ORB%)",
        "Defensive Rebounds (DRB)",
        "Defensive Rebound Percentage (DRB%)",
        "Total Rebounds (TRB)",
        "Total Rebounds Per Game",
        "Total Rebound Percentage (TRB%)",
        "Assists (AST)",
        "Assist Percentage (AST%)",
        "Steals (STL)",
        "Steal Percentage (STL%)",
        "Blocks (BLK)",
        "Block Percentage (BLK%)",
        "Personal Fouls (PF)",
        "Points (PTS)",
        "Turnovers (TOV)",
        "Turnover Percentage (TOV%)",
        "Player Efficiency Rating (PER)",
        # "Game Score (GmSc)",
        # "Offensive Rating (ORtg)",
        # "Defensive Rating (DRtg)",
        "Plus/Minus (+/-)"
]

col_per_game = [
        "ID",
        "Season",
        "Player Name",
        "Team",
        
        "Position",
        "Games Played (G)",
        "Games Started (GS)",
        "Minutes Played (MP)",
        "Field Goals Made (FG)",
        "Field Goal Attempts (FGA)",
        "Field Goal Percentage (FG%)",
        "3-Point Field Goals Made (3P)",
        "3-Point Field Goal Attempts (3PA)",
        "3-Point Field Goal Percentage (3P%)",
        "2-Point Field Goals Made (2P)",
        "2-Point Field Goal Attempts (2PA)",
        "2-Point Field Goal Percentage (2P%)",
        "Effective Field Goal Percentage (eFG%)",
        "Free Throws Made (FT)",
        "Free Throw Attempts (FTA)",
        "Free Throw Percentage (FT%)",
        "Offensive Rebounds (ORB)",
        # "Offensive Rebound Percentage (ORB%)",
        "Defensive Rebounds (DRB)",
        # "Defensive Rebound Percentage (DRB%)",
        "Total Rebounds (TRB)",
        "Total Rebounds Per Game",
        # "Total Rebound Percentage (TRB%)",
        "Assists (AST)",
        # "Assist Percentage (AST%)",
        "Steals (STL)",
        # "Steal Percentage (STL%)",
        "Blocks (BLK)",
        # "Block Percentage (BLK%)",
        "Personal Fouls (PF)",
        "Points (PTS)",
        "Turnovers (TOV)"
        # "Turnover Percentage (TOV%)",
        # "Player Efficiency Rating (PER)",
        # "Game Score (GmSc)",
        # "Offensive Rating (ORtg)",
        # "Defensive Rating (DRtg)",
        # "Plus/Minus (+/-)"
]

col_advanced = [
        # "ID",
        "Season",
        "Player Name",
        "Team",
        
        # "Position",
        # "Games Played (G)",
        # "Games Started (GS)",
        # "Minutes Played (MP)",
        # "Field Goals Made (FG)",
        # "Field Goal Attempts (FGA)",
        # "Field Goal Percentage (FG%)",
        # "3-Point Field Goals Made (3P)",
        # "3-Point Field Goal Attempts (3PA)",
        # "3-Point Field Goal Percentage (3P%)",
        # "2-Point Field Goals Made (2P)",
        # "2-Point Field Goal Attempts (2PA)",
        # "2-Point Field Goal Percentage (2P%)",
        # "Effective Field Goal Percentage (eFG%)",
        # "Free Throws Made (FT)",
        # "Free Throw Attempts (FTA)",
        # "Free Throw Percentage (FT%)",
        # "Offensive Rebounds (ORB)",
        "Offensive Rebound Percentage (ORB%)",
        # "Defensive Rebounds (DRB)",
        "Defensive Rebound Percentage (DRB%)",
        # "Total Rebounds (TRB)",
        # "Total Rebounds Per Game",
        "Total Rebound Percentage (TRB%)",
        # "Assists (AST)",
        "Assist Percentage (AST%)",
        # "Steals (STL)",
        "Steal Percentage (STL%)",
        # "Blocks (BLK)",
        "Block Percentage (BLK%)",
        # "Personal Fouls (PF)",
        # "Points (PTS)",
        # "Turnovers (TOV)",
        "Turnover Percentage (TOV%)",
        "Player Efficiency Rating (PER)",
        # "Game Score (GmSc)",
        # "Offensive Rating (ORtg)",
        # "Defensive Rating (DRtg)",
        "Plus/Minus (+/-)"
]


In [14]:
def get_info(url: str, headers: dict, columns: list, start_id: int):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        df = pd.DataFrame(columns = col_per_game)
        
        while True:
                try:
                        info_tag = soup.select("#per_game_stats > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        
        lst = info_tag[0].select("tr")
        info = [row for row in lst if (row.get("class") == None or "partial_table" in row.get("class"))]
        
        data_stat = [
                "pos",
                "games",
                "games_started",
                "mp_per_g",
                "fg_per_g",
                "fga_per_g",
                "fg_pct",
                "fg3_per_g",
                "fg3a_per_g",
                "fg3_pct",
                "fg2_per_g",
                "fg2a_per_g",
                "fg2_pct",
                "efg_pct",
                "ft_per_g",
                "fta_per_g",
                "ft_pct",
                "orb_per_g",
                "drb_per_g",
                "trb_per_g",
                "trb_per_g",
                "ast_per_g",
                "stl_per_g",
                "blk_per_g",
                "pf_per_g",
                "pts_per_g",
                "tov_per_g",
        ]
        
        for row in info:
                result = {}
                
                result[columns[0]] = len(df) + start_id
                # Season
                try:
                        # "https://www.basketball-reference.com/leagues/NBA_1982_per_game.html"
                        result[columns[1]] = url[49: 53]
                except:
                        pass
                
                # Player Name
                try:
                        result[columns[2]] = row.find(attrs={"data-stat": "name_display"}).a.text.strip()
                except:
                        pass
                # Team
                try:
                        result[columns[3]] = row.find(attrs={"data-stat": "team_name_abbr"}).a.text.strip()
                except:
                        try:
                                result["Team"] = row.find(attrs={"data-stat": "team_name_abbr"}).text.strip()
                        except:
                                pass
                        
                for f in range(len(data_stat)):
                        if (data_stat[f] == ""):
                                continue
                        try:
                                result[col_per_game[f + 4]] = row.find(attrs={"data-stat": data_stat[f]}).text.strip()
                        except:
                                try:
                                        result[col_per_game[f + 4]] = row.find(attrs={"data-stat": data_stat[f]}).strong.text.strip()
                                except:
                                        pass
                                
                df.loc[len(df)] = result
        
        time.sleep(_Delay)
        return df

In [18]:
player_season_data = pd.DataFrame(columns = col_per_game)
row_id = 0
for i in tqdm(range(len(season_url))):
        result = get_info(season_url[i], headers, columns, row_id)
        row_id += len(result)
        
        player_season_data = pd.concat([player_season_data, result], ignore_index = True)
        player_season_data.to_csv("../../data/raw/player_season_data.csv", index = False)
        
player_season_data.head()

100%|██████████| 6/6 [00:42<00:00,  7.14s/it]


,ID,Season,Player Name,Team,Position,Games Played (G),Games Started (GS),Minutes Played (MP),Field Goals Made (FG),Field Goal Attempts (FGA),...,Offensive Rebounds (ORB),Defensive Rebounds (DRB),Total Rebounds (TRB),Total Rebounds Per Game,Assists (AST),Steals (STL),Blocks (BLK),Personal Fouls (PF),Points (PTS),Turnovers (TOV)
0,0,2020,James Harden,HOU,SG,68,68,36.5,9.9,22.3,...,1.0,5.5,6.6,6.6,7.5,1.8,0.9,3.3,34.3,4.5
1,1,2020,Bradley Beal,WAS,SG,57,57,36.0,10.4,22.9,...,0.9,3.3,4.2,4.2,6.1,1.2,0.4,2.2,30.5,3.4
2,2,2020,Damian Lillard,POR,PG,66,66,37.5,9.5,20.4,...,0.5,3.8,4.3,4.3,8.0,1.1,0.3,1.7,30.0,2.9
3,3,2020,Trae Young,ATL,PG,60,60,35.3,9.1,20.8,...,0.5,3.7,4.3,4.3,9.3,1.1,0.1,1.7,29.6,4.8
4,4,2020,Giannis Antetokounmpo,MIL,PF,63,63,30.4,10.9,19.7,...,2.2,11.4,13.6,13.6,5.6,1.0,1.0,3.1,29.5,3.7


In [19]:
with open("../../data/raw/NBA_advanced.txt", "r") as f:
        season_advanced_url = f.read().splitlines()
season_advanced_url[:5]

player_season_advanced_data = pd.DataFrame(columns = col_advanced)    

In [20]:
def get_advanced_info(url: str, headers: dict, columns: list):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        df = pd.DataFrame(columns = col_advanced)
        
        while True:
                try:
                        info_tag = soup.select("#advanced > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        
        lst = info_tag[0].select("tr")
        info = [row for row in lst if (row.get("class") == None or "partial_table" in row.get("class"))]
        
        data_stat = [
                "orb_pct",
                "drb_pct",
                "trb_pct",
                "ast_pct",
                "stl_pct",
                "blk_pct",
                "tov_pct",
                "per",
                "bpm"
        ]
        
        for row in info:
                result = {}
                
                # result[columns[0]] = len(df) + start_id
                # Season
                try:
                        # "https://www.basketball-reference.com/leagues/NBA_1982_per_game.html"
                        result[columns[1]] = int(url[49: 53])
                except:
                        pass
                
                # Player Name
                try:
                        result[columns[2]] = row.find(attrs={"data-stat": "name_display"}).a.text.strip()
                except:
                        pass
                # Team
                try:
                        result[columns[3]] = row.find(attrs={"data-stat": "team_name_abbr"}).a.text.strip()
                except:
                        try:
                                result["Team"] = row.find(attrs={"data-stat": "team_name_abbr"}).text.strip()
                        except:
                                pass
                        
                for f in range(len(data_stat)):
                        if (data_stat[f] == ""):
                                continue
                        # print(col_advanced[f])
                        try:
                                result[col_advanced[f + 3]] = row.find(attrs={"data-stat": data_stat[f]}).text.strip()
                        except:
                                try:
                                        result[col_advanced[f + 3]] = row.find(attrs={"data-stat": data_stat[f]}).strong.text.strip()
                                except:
                                        pass
                                
                df.loc[len(df)] = result
        
        time.sleep(_Delay)
        return df

In [21]:
for i in tqdm(range(len(season_advanced_url))):
        result = get_advanced_info(season_advanced_url[i], headers, columns)
        
        player_season_advanced_data = pd.concat([player_season_advanced_data, result], ignore_index = True)
        
player_season_data['Season'] = player_season_data['Season'].astype("int")
player_season_advanced_data['Season'] = player_season_advanced_data['Season'].astype("int")
        
df_merge = player_season_data.merge(
        player_season_advanced_data,
        on=columns[1:4],
        how='left')
df_merge = df_merge[columns]
df_merge.to_csv("../../data/raw/player_season_data.csv", index = False)

100%|██████████| 6/6 [00:39<00:00,  6.54s/it]
